In [ ]:
from google.colab import files
uploaded = files.upload()


Saving LoanApprovalPrediction.csv to LoanApprovalPrediction.csv


In [ ]:
import pandas as pd

df = pd.read_csv('LoanApprovalPrediction.csv')
print("Shape of dataset:", df.shape)
print(df.head())
print(df.info())
print("Missing values:\n", df.isnull().sum())


Shape of dataset: (598, 13)
    Loan_ID Gender Married  Dependents     Education Self_Employed  \
0  LP001002   Male      No         0.0      Graduate            No   
1  LP001003   Male     Yes         1.0      Graduate            No   
2  LP001005   Male     Yes         0.0      Graduate           Yes   
3  LP001006   Male     Yes         0.0  Not Graduate            No   
4  LP001008   Male      No         0.0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural 

In [ ]:
# Fill missing categorical features with mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History', 'Loan_Amount_Term']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fill LoanAmount with median
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)

# Drop Loan_ID (not useful for prediction)
df.drop('Loan_ID', axis=1, inplace=True)

# Convert 'Dependents' from '3+' to 3
df['Dependents'].replace('3+', 3, inplace=True)
df['Dependents'] = df['Dependents'].astype(int)

# Encode binary categorical variables
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Loan_Status']
for col in binary_cols:
    df[col] = df[col].map({'Male':1, 'Female':0, 'Yes':1, 'No':0, 'Graduate':1, 'Not Graduate':0, 'Y':1, 'N':0})

# One-hot encode Property_Area
df = pd.get_dummies(df, columns=['Property_Area'], drop_first=True)

# Create new features
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['DTI'] = df['LoanAmount'] / df['TotalIncome']
df['LTI'] = df['LoanAmount'] / df['ApplicantIncome']


<ipython-input-5-ccf7990d8f05>:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
<ipython-input-5-ccf7990d8f05>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using '

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define features and target
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f'{name} Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, zero_division=0))


Logistic Regression Accuracy: 0.8250
              precision    recall  f1-score   support

           0       0.94      0.43      0.59        35
           1       0.81      0.99      0.89        85

    accuracy                           0.82       120
   macro avg       0.87      0.71      0.74       120
weighted avg       0.85      0.82      0.80       120

Decision Tree Accuracy: 0.7167
              precision    recall  f1-score   support

           0       0.51      0.60      0.55        35
           1       0.82      0.76      0.79        85

    accuracy                           0.72       120
   macro avg       0.67      0.68      0.67       120
weighted avg       0.73      0.72      0.72       120

Random Forest Accuracy: 0.8333
              precision    recall  f1-score   support

           0       0.86      0.51      0.64        35
           1       0.83      0.96      0.89        85

    accuracy                           0.83       120
   macro avg       0.84      

In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.4 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import joblib

# Evaluate models using cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-validation results:")
best_model = None
best_score = 0
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    avg_score = np.mean(scores)
    print(f'{name}: {avg_score:.4f}')
    if avg_score > best_score:
        best_score = avg_score
        best_model = model

# Fit best model on full training data
best_model.fit(X_train_scaled, y_train)

# Save model and scaler
joblib.dump(best_model, 'loan_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'features.pkl')



Cross-validation results:
Logistic Regression: 0.8034
Decision Tree: 0.7199
Random Forest: 0.7909
Gradient Boosting: 0.7887


['features.pkl']

In [ ]:
import gradio as gr
import numpy as np

# Load model, scaler, and features
model = joblib.load('loan_model.pkl')
scaler = joblib.load('scaler.pkl')
feature_names = joblib.load('features.pkl')

def predict_loan(Gender, Married, Dependents, Education, Self_Employed, ApplicantIncome, CoapplicantIncome,
                 LoanAmount, Loan_Amount_Term, Credit_History, Property_Area):

    TotalIncome = ApplicantIncome + CoapplicantIncome
    DTI = LoanAmount / TotalIncome
    LTI = LoanAmount / (ApplicantIncome if ApplicantIncome > 0 else 1)

    # Binary encoding
    Gender = 1 if Gender == 'Male' else 0
    Married = 1 if Married == 'Yes' else 0
    Education = 1 if Education == 'Graduate' else 0
    Self_Employed = 1 if Self_Employed == 'Yes' else 0

    # One-hot for Property_Area
    PA_Semiurban = 1 if Property_Area == 'Semiurban' else 0
    PA_Urban = 1 if Property_Area == 'Urban' else 0

    # Feature vector
    data = [Gender, Married, Dependents, Education, Self_Employed,
            ApplicantIncome, CoapplicantIncome, LoanAmount,
            Loan_Amount_Term, Credit_History, PA_Semiurban, PA_Urban,
            TotalIncome, DTI, LTI]

    # Scale
    data = np.array(data).reshape(1, -1)
    data_scaled = scaler.transform(data)

    # Predict
    pred = model.predict(data_scaled)
    return "Approved ✅" if pred[0] == 1 else "Rejected ❌"

demo = gr.Interface(
    fn=predict_loan,
    inputs=[
        gr.Radio(['Male', 'Female'], label="Gender"),
        gr.Radio(['Yes', 'No'], label="Married"),
        gr.Slider(0, 3, step=1, label="Dependents"),
        gr.Radio(['Graduate', 'Not Graduate'], label="Education"),
        gr.Radio(['Yes', 'No'], label="Self Employed"),
        gr.Number(label="Applicant Income"),
        gr.Number(label="Coapplicant Income"),
        gr.Number(label="Loan Amount"),
        gr.Number(label="Loan Amount Term"),
        gr.Radio([1.0, 0.0], label="Credit History"),
        gr.Radio(['Rural', 'Semiurban', 'Urban'], label="Property Area")
    ],
    outputs="text",
    title="🏦 Loan Approval Predictor",
    description="Enter applicant details below to check if the loan is likely to be approved."
)

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://677381486ab5f2b0b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
